<a href="https://colab.research.google.com/github/humeyragul/CSS/blob/master/HepsiBuradaMVPson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
"""
Lojistik Rota Optimizasyonu ve Talep Tahmini Modülü / TEKNOFEST 2026
"""

import pandas as pd
import numpy as np
import time
import re
import warnings
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
import pulp

warnings.filterwarnings('ignore')

# GLOBAL AYARLAR VE SABİTLER (CONFIG)
CONFIG = {
    'start_date': '2026-05-11',
    'end_date': '2026-05-17',
    'ort_spot_km_maliyet': 20,
    'arac_kapasiteleri': {
        'Tır': 22400,
        'Kamyon': 12000,
        'Hafif Kamyon': 7200,
        'Kamyonet': 5600
    }
}

# YARDIMCI FONKSİYONLAR
def haversine_distance(lat1, lon1, lat2, lon2):
    """İki koordinat arası kuş uçuşu mesafeyi hesaplar."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return round(c * 6371.0, 2)

def extract_vehicle_capacity(arac_str):
    """Araç stringini okuyarak toplam desiyi hesaplar (Örn: '2x Tır + 1x Kamyonet')."""
    if pd.isna(arac_str) or arac_str == "Araç Gerekmedi": return 0
    cap = 0
    for p in str(arac_str).split('+'):
        try:
            adet = int(re.search(r'\d+', p).group())
            for tur, kapasite in CONFIG['arac_kapasiteleri'].items():
                if tur in p:
                    cap += adet * kapasite
        except:
            pass
    return cap

def get_distance(a, b, mesafe_dict):
    """İki merkez arası mesafeyi sözlükten çeker, yoksa 500 km ceza keser."""
    if a == b: return 0
    return mesafe_dict.get((a, b), 500)

# 1. MODÜL: MESAFE HESAPLAMA
def generate_distance_matrix():
    print(" Adım 1: Mesafe Matrisi Oluşturuluyor...")
    df_coords = pd.read_excel('Koordinatlar v2.xlsx')
    mesafe_listesi = []

    for _, row1 in df_coords.iterrows():
        for _, row2 in df_coords.iterrows():
            dist = haversine_distance(row1['Enlem'], row1['Boylam'], row2['Enlem'], row2['Boylam'])
            mesafe_listesi.append({
                'Çıkış Transfer Merkezi': row1['Transfer Merkezi'],
                'Varış Transfer Merkezi': row2['Transfer Merkezi'],
                'Kus_Ucusu_KM': dist
            })

    df_kus_ucusu = pd.DataFrame(mesafe_listesi)
    output_file = 'Kus_Ucusu_Mesafeler.xlsx'
    df_kus_ucusu.to_excel(output_file, index=False)
    print(f"Mesafeler hesaplandı: {output_file}")
    return df_kus_ucusu

# 2. MODÜL: TALEP TAHMİNİ (FORECASTING)
def forecast_demand():
    print(" Adım 2: Geçmiş verilerle talep tahminleniyor...")
    df = pd.read_excel('Desi_talep.xlsx')
    df['Tarih'] = pd.to_datetime(df['Tarih'])
    df['Rota'] = df['Çıkış Transfer Merkezi'] + '_' + df['Varış Transfer Merkezi']

    le = LabelEncoder()
    df['Rota_Encoded'] = le.fit_transform(df['Rota'])
    df['Gun'] = df['Tarih'].dt.day
    df['Haftanin_Gunu'] = df['Tarih'].dt.dayofweek
    df['Ay'] = df['Tarih'].dt.month
    df['Haftasonu'] = df['Haftanin_Gunu'].apply(lambda x: 1 if x >= 5 else 0)

    df = df.sort_values(['Rota', 'Tarih']).reset_index(drop=True)
    df['Lag_7'] = df.groupby('Rota')['Toplam Desi'].shift(7)
    train_df = df.dropna().copy()

    features = ['Rota_Encoded', 'Gun', 'Haftanin_Gunu', 'Ay', 'Haftasonu', 'Lag_7']
    target = 'Toplam Desi'

    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(train_df[features], train_df[target])

    # Gelecek Zaman Matrisi
    hedef_tarihler = pd.date_range(start=CONFIG['start_date'], end=CONFIG['end_date'])
    benzersiz_rotalar = df[['Çıkış Transfer Merkezi', 'Varış Transfer Merkezi', 'Rota', 'Rota_Encoded']].drop_duplicates()

    future_list = [benzersiz_rotalar.assign(Tarih=t) for t in hedef_tarihler]
    future_df = pd.concat(future_list, ignore_index=True)

    future_df['Gun'] = future_df['Tarih'].dt.day
    future_df['Haftanin_Gunu'] = future_df['Tarih'].dt.dayofweek
    future_df['Ay'] = future_df['Tarih'].dt.month
    future_df['Haftasonu'] = future_df['Haftanin_Gunu'].apply(lambda x: 1 if x >= 5 else 0)

    gecmis_talep = df[['Rota', 'Tarih', 'Toplam Desi']].copy()
    gecmis_talep['Hedef_Tarih'] = gecmis_talep['Tarih'] + pd.Timedelta(days=7)

    future_df = pd.merge(future_df, gecmis_talep[['Rota', 'Hedef_Tarih', 'Toplam Desi']],
                         left_on=['Rota', 'Tarih'], right_on=['Rota', 'Hedef_Tarih'], how='left')
    future_df.rename(columns={'Toplam Desi': 'Lag_7'}, inplace=True)
    future_df['Lag_7'] = future_df['Lag_7'].fillna(0)

    future_df['Tahmini_Desi'] = rf_model.predict(future_df[features])
    future_df['Tahmini_Desi'] = future_df['Tahmini_Desi'].apply(lambda x: max(0, round(x, 2)))

    final_output = future_df[['Çıkış Transfer Merkezi', 'Varış Transfer Merkezi', 'Tarih', 'Tahmini_Desi']].copy()
    final_output.rename(columns={'Tahmini_Desi': 'Toplam Desi'}, inplace=True)
    final_output['Tarih'] = final_output['Tarih'].dt.strftime('%Y-%m-%d')

    output_name = 'Tahminlenen_Talep.xlsx'
    final_output.to_excel(output_name, index=False)
    print(f"✅ Tahminler üretildi: {output_name}")

# 3. MODÜL: FAZ 1 (DİREKT ATAMALAR)
def run_phase1_optimization():
    print(" Adım 3: ILP Rota Optimizasyonu Başlıyor (Faz 1)...")
    df_talep = pd.read_excel('Tahminlenen_Talep.xlsx')
    df_mesafe = pd.read_excel('Kus_Ucusu_Mesafeler.xlsx')
    df_kiralik = pd.read_excel('Kiralik_Araclar.xlsx')
    df_maliyet = pd.read_excel('Arac_Kapasite_Maliyet.xlsx')

    df_talep['Rota'] = df_talep['Çıkış Transfer Merkezi'] + '_' + df_talep['Varış Transfer Merkezi']
    df_mesafe['Rota'] = df_mesafe['Çıkış Transfer Merkezi'] + '_' + df_mesafe['Varış Transfer Merkezi']
    df_talep = pd.merge(df_talep, df_mesafe[['Rota', 'Kus_Ucusu_KM']], on='Rota', how='left')

    kiralik_dict = {}
    for _, row in df_kiralik.iterrows():
        rota = row['Çıkış Transfer Merkezi'] + '_' + row['Varış Transfer Merkezi']
        kiralik_dict.setdefault(rota, []).append({'Tur': row['Araç Türü'], 'Adet': row['Araç sayısı']})

    arac_ozellikleri = df_maliyet.set_index('Araç Adı').to_dict(orient='index')

    planlama_sonuclari = []
    ugrama_havuzu = []
    toplam_sistem_maliyeti = 0

    for idx, row in df_talep.iterrows():
        talep_desi = row['Toplam Desi']
        rota = row['Rota']
        km = row['Kus_Ucusu_KM']

        if talep_desi == 0 and rota not in kiralik_dict:
            continue

        kalan_talep = talep_desi
        gunluk_maliyet = 0
        kullanilan_araclar = []

        if rota in kiralik_dict:
            for kiralik in kiralik_dict[rota]:
                tur, adet = kiralik['Tur'], kiralik['Adet']
                kapasite = arac_ozellikleri[tur]['Kapasite (desi)']
                maliyet = (arac_ozellikleri[tur]['Kiralık Araç Günlük Kira (TL)'] + (arac_ozellikleri[tur]['Kiralık Araç Kilometre Başına Maliyet (TL)'] * km)) * adet
                gunluk_maliyet += maliyet
                kalan_talep -= (kapasite * adet)
                kullanilan_araclar.append(f"{adet}x Kiralık {tur}")

        kalan_talep = round(max(0, kalan_talep), 2)
        ugrama_durumu = "Direkt Teslimat"

        if kalan_talep > 0:
            prob = pulp.LpProblem(f"Spot_{idx}", pulp.LpMinimize)
            vars_dict = {
                'Tır': pulp.LpVariable('Tir', lowBound=0, cat='Integer'),
                'Kamyon': pulp.LpVariable('Kamyon', lowBound=0, cat='Integer'),
                'Hafif Kamyon': pulp.LpVariable('Hafif_Kamyon', lowBound=0, cat='Integer'),
                'Kamyonet': pulp.LpVariable('Kamyonet', lowBound=0, cat='Integer')
            }

            maliyet_eq = 0
            kapasite_eq = 0
            for tur, var in vars_dict.items():
                maliyet = arac_ozellikleri[tur]['Spot Araç Sabit Günlük Maliyet (TL)'] + (arac_ozellikleri[tur]['Spot Kilometre Başına Maliyet (TL)'] * km)
                kapasite = arac_ozellikleri[tur]['Kapasite (desi)']
                maliyet_eq += maliyet * var
                kapasite_eq += kapasite * var

            prob += maliyet_eq
            prob += kapasite_eq >= kalan_talep
            prob += (kapasite_eq * 0.1) <= kalan_talep # %10 Kuralı

            prob.solve(pulp.PULP_CBC_CMD(msg=False))

            if prob.status == 1:
                for tur, var in vars_dict.items():
                    if var.varValue > 0:
                        kullanilan_araclar.append(f"{int(var.varValue)}x Spot {tur}")
                        maliyet = arac_ozellikleri[tur]['Spot Araç Sabit Günlük Maliyet (TL)'] + (arac_ozellikleri[tur]['Spot Kilometre Başına Maliyet (TL)'] * km)
                        gunluk_maliyet += maliyet * var.varValue
            else:
                ugrama_durumu = f"{kalan_talep} Desi Uğrama Bekliyor"
                ugrama_havuzu.append({
                    'Tarih': row['Tarih'], 'Çıkış': row['Çıkış Transfer Merkezi'],
                    'Varış': row['Varış Transfer Merkezi'], 'Bekleyen_Desi': kalan_talep
                })

        toplam_sistem_maliyeti += gunluk_maliyet
        planlama_sonuclari.append({
            'Tarih': row['Tarih'], 'Çıkış Transfer Merkezi': row['Çıkış Transfer Merkezi'],
            'Varış Transfer Merkezi': row['Varış Transfer Merkezi'], 'Toplam Desi': talep_desi,
            'Kuş Uçuşu KM': km, 'Atanan Araçlar': " + ".join(kullanilan_araclar) if kullanilan_araclar else "Araç Gerekmedi",
            'Operasyon Durumu': ugrama_durumu, 'Rota Maliyeti (TL)': round(gunluk_maliyet, 2)
        })

    pd.DataFrame(planlama_sonuclari).to_excel('Faz1_Direkt_Atamalar.xlsx', index=False)
    pd.DataFrame(ugrama_havuzu).to_excel('Faz2_Ugrama_Bekleyenler.xlsx', index=False)
    print(f" Faz 1 Tamamlandı. Açıkta kalan yük sayısı: {len(ugrama_havuzu)}")

# 4. MODÜL: FAZ 2 (GLOBAL INSERTION)
def run_phase2_global_insertion():
    print(" Adım 4: Genişletilmiş Uğrama Algoritması Çalıştırılıyor...")
    df_faz1 = pd.read_excel('Faz1_Direkt_Atamalar.xlsx')
    df_faz2 = pd.read_excel('Faz2_Ugrama_Bekleyenler.xlsx')
    df_mesafe = pd.read_excel('Kus_Ucusu_Mesafeler.xlsx')

    mesafe_dict = {(row['Çıkış Transfer Merkezi'], row['Varış Transfer Merkezi']): row['Kus_Ucusu_KM'] for _, row in df_mesafe.iterrows()}

    df_faz1['Toplam_Kapasite'] = df_faz1['Atanan Araçlar'].apply(extract_vehicle_capacity)
    df_faz1['Bos_Kapasite'] = df_faz1['Toplam_Kapasite'] - df_faz1['Toplam Desi']
    df_faz1['Ugrama_Detayi'] = ""

    basarili_ugrama = 0

    for idx, row in df_faz2.iterrows():
        tarih, A, B, desi = row['Tarih'], row['Çıkış'], row['Varış'], row['Bekleyen_Desi']

        adaylar = df_faz1[(df_faz1['Tarih'] == tarih) & (df_faz1['Bos_Kapasite'] >= desi) & (df_faz1['Atanan Araçlar'] != "Araç Gerekmedi")].copy()

        if not adaylar.empty:
            adaylar['Ekstra_KM'] = adaylar.apply(
                lambda r: (get_distance(r['Çıkış Transfer Merkezi'], A, mesafe_dict) +
                           get_distance(A, B, mesafe_dict) +
                           get_distance(B, r['Varış Transfer Merkezi'], mesafe_dict)) -
                           get_distance(r['Çıkış Transfer Merkezi'], r['Varış Transfer Merkezi'], mesafe_dict), axis=1)

            best_idx = adaylar['Ekstra_KM'].idxmin()
            df_faz1.loc[best_idx, 'Bos_Kapasite'] -= desi

            yeni_not = f"{A}->{B} yükünü aldı (+{desi} Desi)"
            mevcut_not = df_faz1.loc[best_idx, 'Ugrama_Detayi']
            df_faz1.loc[best_idx, 'Ugrama_Detayi'] = mevcut_not + " | " + yeni_not if mevcut_not else yeni_not
            df_faz1.loc[best_idx, 'Rota Maliyeti (TL)'] += adaylar.loc[best_idx, 'Ekstra_KM'] * CONFIG['ort_spot_km_maliyet']
            basarili_ugrama += 1

    df_nihai = df_faz1[['Tarih', 'Çıkış Transfer Merkezi', 'Varış Transfer Merkezi', 'Toplam Desi', 'Kuş Uçuşu KM', 'Atanan Araçlar', 'Rota Maliyeti (TL)', 'Ugrama_Detayi']]
    df_nihai.to_excel('Nihai_Teknofest_Sifir_Hata.xlsx', index=False)

    # --- EKSİK OLAN KISIM BURASIYDI: EKRANA YAZDIRMA ---
    nihai_maliyet = df_nihai['Rota Maliyeti (TL)'].sum()
    toplam_desi = df_nihai['Toplam Desi'].sum()

    print(" FİNAL OPERASYON RAPORU (JÜRİ ÖZETİ)")
    print(f" Taşınan Toplam Hacim : {toplam_desi:,.2f} Desi")
    print(f" TOPLAM SİSTEM MALİYETİ: {nihai_maliyet:,.2f} TL")
    print(f" Başarılı Uğrama Sayısı: {basarili_ugrama} / {len(df_faz2)}")
    print("Dosya 'Nihai_Teknofest_Sifir_Hata.xlsx' olarak kaydedildi.")

# 5. MODÜL: JÜRİ TESLİM FORMATINA DÖNÜŞTÜRME
def format_for_submission():
    print("\n Jüri teslim formatları hazırlanıyor...")

    # 1. TAHMİN DOSYASI DÜZENLEMESİ
    df_tahmin = pd.read_excel('Tahminlenen_Talep.xlsx')
    df_tahmin = df_tahmin.rename(columns={
        'Çıkış Transfer Merkezi': 'Çıkış TM',
        'Varış Transfer Merkezi': 'Varış TM',
        'Toplam Desi': 'Tahmin Edilen Desi'
    })
    # Sütun sırasını ayarlama
    df_tahmin = df_tahmin[['Tarih', 'Çıkış TM', 'Varış TM', 'Tahmin Edilen Desi']]
    df_tahmin.to_excel('Tahmin_Teslim.xlsx', index=False)

    # 2. ARAÇ PLANLAMA DOSYASI DÜZENLEMESİ (Ayrıştırma İşlemi)
    df_plan = pd.read_excel('Nihai_Teknofest_Sifir_Hata.xlsx')
    teslim_satirlari = []

    for _, row in df_plan.iterrows():
        araclar_str = str(row['Atanan Araçlar'])
        if araclar_str == "Araç Gerekmedi" or pd.isna(araclar_str):
            continue

        # Toplam rotadaki desi ve maliyeti araç sayısına oransal dağıtıyoruz (Basitleştirilmiş yaklaşım)
        parcalar = araclar_str.split('+')
        toplam_arac_sayisi = sum([int(re.search(r'\d+', p).group()) for p in parcalar if re.search(r'\d+', p)])

        for p in parcalar:
            match = re.search(r'(\d+)x\s+(.*)', p.strip())
            if match:
                adet = int(match.group(1))
                arac_tipi = match.group(2).strip()

                # Her bir araç için ayrı satır oluştur (Maliyet ve Desi'yi araca bölüştürüyoruz)
                for _ in range(adet):
                    teslim_satirlari.append({
                        'Tarih': row['Tarih'],
                        'Araç Tipi': arac_tipi,
                        'Çıkış TM': row['Çıkış Transfer Merkezi'].replace('Transfer Merkezi', 'TM').strip(),
                        'Varış TM': row['Varış Transfer Merkezi'].replace('Transfer Merkezi', 'TM').strip(),
                        'Atanan Desi': round(row['Toplam Desi'] / toplam_arac_sayisi, 2), # Yükü araçlara böl
                        'Maliyet': round(row['Rota Maliyeti (TL)'] / toplam_arac_sayisi, 2) # Maliyeti araçlara böl
                    })

    df_teslim_plan = pd.DataFrame(teslim_satirlari)
    df_teslim_plan = df_teslim_plan[['Tarih', 'Araç Tipi', 'Çıkış TM', 'Varış TM', 'Atanan Desi', 'Maliyet']]
    df_teslim_plan.to_excel('Arac_Planlama_Teslim.xlsx', index=False)
    print("✅ Teslim dosyaları 'Tahmin_Teslim.xlsx' ve 'Arac_Planlama_Teslim.xlsx' olarak oluşturuldu!")

# ANA ÇALIŞTIRMA BLOĞU
if __name__ == "__main__":
    print("="*50)
    print("TEKNOFEST LOJİSTİK OPTİMİZASYON SİSTEMİ BAŞLATILDI")
    print("="*50)

    # Adımları sırasıyla çağırıyoruz
    generate_distance_matrix()
    forecast_demand()
    run_phase1_optimization()
    run_phase2_global_insertion()
    format_for_submission()

    print("="*50)
    print("Tüm işlemler başarıyla ve Clean Code standartlarında tamamlandı!")

TEKNOFEST LOJİSTİK OPTİMİZASYON SİSTEMİ BAŞLATILDI
 Adım 1: Mesafe Matrisi Oluşturuluyor...
Mesafeler hesaplandı: Kus_Ucusu_Mesafeler.xlsx
 Adım 2: Geçmiş verilerle talep tahminleniyor...
✅ Tahminler üretildi: Tahminlenen_Talep.xlsx
 Adım 3: ILP Rota Optimizasyonu Başlıyor (Faz 1)...
 Faz 1 Tamamlandı. Açıkta kalan yük sayısı: 58
 Adım 4: Genişletilmiş Uğrama Algoritması Çalıştırılıyor...
 FİNAL OPERASYON RAPORU (JÜRİ ÖZETİ)
 Taşınan Toplam Hacim : 8,281,339.92 Desi
 TOPLAM SİSTEM MALİYETİ: 11,133,979.77 TL
 Başarılı Uğrama Sayısı: 58 / 58
Dosya 'Nihai_Teknofest_Sifir_Hata.xlsx' olarak kaydedildi.

 Jüri teslim formatları hazırlanıyor...
✅ Teslim dosyaları 'Tahmin_Teslim.xlsx' ve 'Arac_Planlama_Teslim.xlsx' olarak oluşturuldu!
Tüm işlemler başarıyla ve Clean Code standartlarında tamamlandı!
